In [ ]:

from peft import PeftModel, PeftConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

: 

In [2]:
# Trained adapter path
peft_model_id = "/home/jovyan/Works/MSc_Thesis/mh_conversation/finetuned_model/llama3.2_3b_mh_conversation_v7" 

print("Loading model...")

# Load PEFT config to get base model info
config = PeftConfig.from_pretrained(peft_model_id)
print(f"Base model: {config.base_model_name_or_path}")

# Load base model
model = AutoModelForCausalLM.from_pretrained(
    config.base_model_name_or_path,
    device_map={"": 1},
    torch_dtype=torch.bfloat16
)

# Load tokenizer from the adapter (contains special tokens)
tokenizer = AutoTokenizer.from_pretrained(peft_model_id)
print(f"Tokenizer vocab size: {len(tokenizer)}")

# Resize model to accommodate new tokens
model.resize_token_embeddings(len(tokenizer))

# Load the adapter weights
model = PeftModel.from_pretrained(model, peft_model_id)
model.to(torch.bfloat16)
model.eval()

print("Model loaded successfully!")

Loading model...
Base model: meta-llama/Llama-3.2-3B-Instruct


Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.98it/s]


Tokenizer vocab size: 128256


/home/jovyan/Works/MSc_Thesis/mh_venv/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:693: UserWarning: Model with `tie_word_embeddings=True` and the tied_target_modules=['model.embed_tokens', 'lm_head'] are part of the adapter. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. See for example https://github.com/huggingface/peft/issues/2018.
  warnings.warn(


Model loaded successfully!


In [3]:
# Test the model with the CORRECT tokenizer (the one with special tokens)
def test_model(patient_id, query, max_length=1024):
    # Format exactly like training data - include the explicit instruction
    messages = [
        # {
        #     "role": "system",
        #     "content": "You are a compassionate mental health support assistant who responds in a warm, empathetic, and non-judgmental way, validating the user’s feelings, offering practical and evidence-based coping suggestions, encouraging professional help when appropriate, providing crisis resources only if there is self-harm or suicide risk, and tailoring hopeful guidance to the user’s specific situation."
        # },
        {
            "role": "user",
            "content": f"Query: {query}"
        }
    ]
    
    # Apply chat template using the LOADED tokenizer
    prompt = tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
    )
    
    # Tokenize and generate
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_length = inputs['input_ids'].shape[1]
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_length,
            do_sample=True,
            # temperature=0.01,              
            # top_p=0.95,                 
            # repetition_penalty=1.0,     
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            
            
        )
    
    response_tokens = outputs[0][input_length:]
    response = tokenizer.decode(response_tokens, skip_special_tokens=False)
    
    # Clean up the eos for clear response
    response = response.replace(tokenizer.eos_token, "").strip()
    
    return response

In [6]:
import textwrap

test_query = "I am feeling like I am worthless"
response = test_model("UID_123", test_query)

# Wrap text at 80 characters per line
print("Response\n")
print("\n".join(textwrap.wrap(response, width=100)))
# print("=" * 80)


Response

I am so sorry to hear you are feeling this way. Feeling worthless can be extremely debilitating.
First of all, these feelings are common and usually temporary. Many people go through periods in
their lives when they feel worthless for short periods of time, but usually there is an underlying
cause for these feelings. In the short term, sometimes it helps to remind yourself that almost
everyone feels insecure at times. Perhaps the best way of getting through days in which you feel
insecure is to remind yourself that these feelings are temporary and will pass. You may also benefit
from focusing on what makes you feel secure and at ease. What helps you feel at peace with yourself?
What outside circumstances do you need in your life to feel secure? If this is a more persistent
problem, which is having you feeling insecure, you may benefit from talking with a therapist. A
therapist can help you to figure the underlying causes of these feelings and help you to learn new
ways of thi